In [ ]:
# @title 1. Cài đặt thư viện xử lý ngôn ngữ
!pip install pyvi

import json
import os
import re
import hashlib
import unicodedata
from pyvi import ViTokenizer
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
print("✅ Đã cài đặt môi trường và kết nối Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã cài đặt môi trường và kết nối Drive!


In [ ]:
# @title 2. Cấu hình File Input/Output
# --- SỬA ĐƯỜNG DẪN TẠI ĐÂY ---
INPUT_FILE = '/content/drive/MyDrive/voz_merged.jsonl'  # Đường dẫn file gốc của bạn
OUTPUT_FILE = '/content/drive/MyDrive/SEG301_Project/data_clean/voz_clean.jsonl' # Đường dẫn file sau khi làm sạch

# Tạo thư mục output nếu chưa có
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

print(f"📂 Input: {INPUT_FILE}")
print(f"📂 Output: {OUTPUT_FILE}")

📂 Input: /content/drive/MyDrive/voz_merged.jsonl
📂 Output: /content/drive/MyDrive/SEG301_Project/data_clean/voz_clean.jsonl


In [ ]:
# @title 3. Định nghĩa hàm Cleaning & Tokenization

def clean_and_tokenize(text):
    if not text:
        return ""

    # 1. Chuẩn hóa Unicode (Fix lỗi font, tổ hợp/dựng sẵn)
    text = unicodedata.normalize('NFC', text)

    # 2. Loại bỏ thẻ HTML và Script rác
    # Xóa script/style
    text = re.sub(r'<(script|style).*?>.*?</\1>', '', text, flags=re.DOTALL)
    # Xóa thẻ HTML còn lại (VD: <b>, <br>, <div>)
    text = re.sub(r'<.*?>', ' ', text)

    # 3. Xử lý rác (Giữ lại Tiếng Việt, Tiếng Anh, số và dấu câu cơ bản)
    # Loại bỏ URL
    text = re.sub(r'http\S+', '', text)
    # Loại bỏ Email
    text = re.sub(r'\S+@\S+', '', text)
    # (Tùy chọn) Chuyển về chữ thường để Index hiệu quả hơn
    text = text.lower()

    # Loại bỏ khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()

    # 4. Tách từ tiếng Việt (Word Segmentation) bằng PyVi
    # Output: "sinh viên đại học fpt" -> "sinh_viên đại_học fpt"
    tokenized_text = ViTokenizer.tokenize(text)

    return tokenized_text

# Test thử
sample = "Học lập   trình <b>Python</b> tại    FPT University! &lt;script&gt;alert(1)&lt;/script&gt;"
print(f"Gốc: {sample}")
print(f"Sạch: {clean_and_tokenize(sample)}")

Gốc: Học lập   trình <b>Python</b> tại    FPT University! &lt;script&gt;alert(1)&lt;/script&gt;
Sạch: học lập_trình python tại fpt university ! & lt ; script & gt ; alert ( 1 ) & lt ; / script & gt ;


In [ ]:
# @title 4. Chạy Pipeline: Đọc -> De-duplicate -> Clean -> Ghi

def process_data_pipeline(input_path, output_path):
    if not os.path.exists(input_path):
        print("❌ Lỗi: Không tìm thấy file Input!")
        return

    print("🚀 Bắt đầu xử lý dữ liệu...")

    # Tập hợp chứa mã hash của nội dung để kiểm tra trùng lặp
    seen_hashes = set()

    count_in = 0
    count_out = 0
    count_dup = 0

    with open(input_path, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:

        for line in f_in:
            count_in += 1
            if count_in % 10000 == 0:
                print(f"...Đang xử lý dòng thứ {count_in}")

            try:
                # 1. Parse JSON
                doc = json.loads(line)

                # Lấy trường nội dung chính (text hoặc content)
                raw_text = doc.get('text') or doc.get('content', '')

                # Bỏ qua nếu nội dung quá ngắn
                if len(raw_text) < 10:
                    continue

                # 2. De-duplication (Loại bỏ trùng lặp)
                # Tạo mã hash MD5 của văn bản gốc
                text_hash = hashlib.md5(raw_text.encode('utf-8')).hexdigest()

                if text_hash in seen_hashes:
                    count_dup += 1
                    continue # Bỏ qua dòng này vì đã gặp rồi

                seen_hashes.add(text_hash)

                # 3. Clean & Tokenize
                clean_text = clean_and_tokenize(raw_text)

                # Bỏ qua nếu sau khi clean không còn gì (toàn ký tự đặc biệt)
                if len(clean_text) < 5:
                    continue

                # 4. Cập nhật lại doc và ghi xuống file
                doc['text_clean'] = clean_text # Lưu vào trường mới hoặc đè lên trường cũ

                # Có thể xóa trường raw để giảm dung lượng file
                # del doc['text']

                f_out.write(json.dumps(doc, ensure_ascii=False) + '\n')
                count_out += 1

            except Exception as e:
                # Bỏ qua các dòng lỗi JSON format
                continue

    print("-" * 30)
    print(f"✅ HOÀN TẤT!")
    print(f"📊 Tổng số dòng input: {count_in}")
    print(f"🗑️ Số dòng trùng lặp (Deleted): {count_dup}")
    print(f"💾 Số dòng sạch được lưu: {count_out}")
    print(f"📂 File kết quả: {output_path}")

# Chạy hàm
process_data_pipeline(INPUT_FILE, OUTPUT_FILE)

🚀 Bắt đầu xử lý dữ liệu...
...Đang xử lý dòng thứ 10000
...Đang xử lý dòng thứ 20000
...Đang xử lý dòng thứ 30000
...Đang xử lý dòng thứ 40000
...Đang xử lý dòng thứ 50000
...Đang xử lý dòng thứ 60000
...Đang xử lý dòng thứ 70000
...Đang xử lý dòng thứ 80000
...Đang xử lý dòng thứ 90000
...Đang xử lý dòng thứ 100000
...Đang xử lý dòng thứ 110000
...Đang xử lý dòng thứ 120000
...Đang xử lý dòng thứ 130000
...Đang xử lý dòng thứ 140000
...Đang xử lý dòng thứ 150000
...Đang xử lý dòng thứ 160000
...Đang xử lý dòng thứ 170000
...Đang xử lý dòng thứ 180000
...Đang xử lý dòng thứ 190000
...Đang xử lý dòng thứ 200000
...Đang xử lý dòng thứ 210000
...Đang xử lý dòng thứ 220000
...Đang xử lý dòng thứ 230000
...Đang xử lý dòng thứ 240000
...Đang xử lý dòng thứ 250000
...Đang xử lý dòng thứ 260000
...Đang xử lý dòng thứ 270000
...Đang xử lý dòng thứ 280000
...Đang xử lý dòng thứ 290000
...Đang xử lý dòng thứ 300000
...Đang xử lý dòng thứ 310000
...Đang xử lý dòng thứ 320000
...Đang xử lý dòng thứ

In [ ]:
# @title 5. Kiểm tra mẫu dữ liệu sạch
print("--- 5 Dòng dữ liệu sau khi xử lý ---")
try:
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 5: break
            data = json.loads(line)
            # In ra trường text_clean
            print(f"[{i+1}] {data.get('text_clean', '')[:200]}...")
except:
    print("Chưa có file output.")

--- 5 Dòng dữ liệu sau khi xử lý ---
[1] mình có tài_khoản từ hồi 2021 nhưng cũng mới dùng voz nhiều hơn dạo gần đây . tuy_vậy , mình nhận thấy rằng có rất nhiều người trên voz rất là kiểu 4chan , chỉ hơi văn_minh hơn một_chút ... ví_dụ new ...
[2] 4chan tệ lắm à , hay anh nghe báo mạng nói thế , 4chan nó đẻ ra voz này còn được . mấy kiến_thức hay zeitgeist mà anh thấy từ trên facebook với x cũng đổ từ 4chan mà thôi . thêm nữa , trong cái thời_đ...
[3] nội_quy box chuyện_trò linh_tinh ™ ( chuyện_trò linh_tinh ™ ( ) ​ mục_đích của box : box chuyện_trò linh_tinh ™ ( sau đây sẽ gọi tắt là ctlt hoặc f17 ) là mục để các thành_viên vozforums bàn về những ...
[4] lưu_ý về vấn_đề post bài sai box : hiện_tại , diễn_đàn voz . vn đã có gần như là đầy_đủ các box về các chủ_đề công_nghệ giống như bên diễn_đàn voz cũ . các bạn lưu_ý giúp , box f17 ( chuyện_trò linh_t...
[5] loser rãnh rỗi sinh nỗi nổi ngồi liệt_kê những người mình có_thể mời trong tương_lai chắc hông quá được 2 bàn quá , mà đó đã ph

In [ ]:
# @title 6. Ép buộc đồng bộ dữ liệu (Force Sync)
import os

# Đường dẫn file output của bạn
FILE_TO_SYNC = OUTPUT_FILE

print("⏳ Đang ép buộc đồng bộ dữ liệu xuống ổ cứng...")

# 1. Đảm bảo file được flush từ Python buffer xuống OS buffer
with open(FILE_TO_SYNC, 'a') as f:
    f.flush()
    os.fsync(f.fileno())

# 2. Kiểm tra lại số dòng thực tế ĐANG CÓ trên Colab (trước khi tải về)
count_check = 0
with open(FILE_TO_SYNC, 'r', encoding='utf-8') as f:
    for _ in f:
        count_check += 1

print(f"✅ Số dòng thực tế đang nằm trên Colab: {count_check}")

if count_check == 1109949:
    print("🎉 Dữ liệu đã đầy đủ! Bạn có thể tải về an toàn.")
    print("Mẹo: Hãy đợi khoảng 1-2 phút sau khi thấy thông báo này rồi hãy vào Drive tải.")
else:
    print(f"⚠️ Vẫn còn lệch {1109949 - count_check} dòng. Hãy chạy lại cell này hoặc đợi thêm vài phút.")

⏳ Đang ép buộc đồng bộ dữ liệu xuống ổ cứng...
✅ Số dòng thực tế đang nằm trên Colab: 1109949
🎉 Dữ liệu đã đầy đủ! Bạn có thể tải về an toàn.
Mẹo: Hãy đợi khoảng 1-2 phút sau khi thấy thông báo này rồi hãy vào Drive tải.
